# Multilevel Analysis Reproducibility Notebook

This notebook prepares and runs the multilevel models used for the capstone report. It is designed to reproduce the null model, full random-intercept model, random-slope comparison, and empirical Bayes state residuals used for Tables 1 and 2 and Figure 9.

Expected input file: `data/processed/hmda_2023_analytical.parquet`

Expected state covariate file if those columns are not already in the analytical dataset: `data/external/state_covariates.csv`

The exact GLMM fit is executed through R/lme4 because it provides a standard likelihood-based multilevel logistic regression implementation.

In [ ]:
from pathlib import Path
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in {"multilevel", "evaluation", "fairness", "models", "eda"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATHS = [
    PROJECT_ROOT / "data" / "processed" / "hmda_2023_analytical.parquet",
    PROJECT_ROOT / "data" / "processed" / "hmda_2023_analytical.csv"
]
STATE_COVARIATES_PATH = PROJECT_ROOT / "data" / "external" / "state_covariates.csv"
OUT_DIR = PROJECT_ROOT / "outputs" / "multilevel"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

## 1. Load the analytical dataset

The dataset should already be cleaned to the report specification: target derived from `action_taken`, missing income and loan amount removed, the 13 analytical variables retained, and continuous predictors prepared consistently with the final report.

In [ ]:
def load_table(paths):
    for path in paths:
        if path.exists() and path.suffix == ".parquet":
            return pd.read_parquet(path), path
        if path.exists() and path.suffix == ".csv":
            return pd.read_csv(path), path
    raise FileNotFoundError("Place hmda_2023_analytical.parquet or hmda_2023_analytical.csv in data/processed/.")

df, data_path = load_table(DATA_PATHS)
print(f"Loaded {data_path}")
print(f"Rows: {len(df):,}")
df.head()

In [ ]:
rename_map = {
    "derived_sex": "applicant_sex",
    "sex": "applicant_sex",
    "dti": "debt_to_income_ratio",
    "ltv": "loan_to_value_ratio"
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

required = [
    "target", "state_code", "debt_to_income_ratio", "loan_amount", "income",
    "loan_to_value_ratio", "loan_purpose", "loan_type", "applicant_sex",
    "occupancy_type", "lien_status", "applicant_age"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

if len(df) != 332301:
    print(f"Warning: expected 332,301 rows from the report, found {len(df):,} rows.")
else:
    print("Sample size matches the report: 332,301 rows.")

## 2. Prepare model variables

The multilevel section models approval log-odds. The evaluation notebook later flips the positive class to denied when computing denied-class PR-AUC.

In [ ]:
def to_numeric_series(series):
    return pd.to_numeric(series.astype(str).str.replace("%", "", regex=False).str.replace(",", "", regex=False), errors="coerce")

def map_dti_value(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    s = str(value).strip().lower().replace(" ", "")
    replacements = {
        "<20%": 1, "<20": 1, "lessthan20%": 1,
        "20%-<30%": 2, "20-30%": 2, "20%to<30%": 2, "20%-30%": 2,
        "30%-<36%": 3, "30-36%": 3, "30%-36%": 3,
        "36%-<41%": 4, "36-41%": 4, "36%-41%": 4,
        "42%-<49%": 5, "42-49%": 5, "42%-49%": 5,
        "50%-60%": 6, "50-60%": 6,
        ">60%": 7, ">60": 7, "greaterthan60%": 7,
        "exempt": np.nan, "na": np.nan, "nan": np.nan
    }
    if s in replacements:
        return replacements[s]
    return pd.to_numeric(s.replace("%", ""), errors="coerce")

model_df = df.copy()
model_df["target_approved"] = pd.to_numeric(model_df["target"], errors="coerce").astype(int)
model_df["state_code"] = model_df["state_code"].astype(str).str.upper().str.strip()
model_df["dti_ordinal"] = model_df["debt_to_income_ratio"].apply(map_dti_value)
model_df["dti_missing"] = model_df["dti_ordinal"].isna().astype(int)
model_df["dti_ordinal"] = model_df["dti_ordinal"].fillna(model_df["dti_ordinal"].median())
model_df["loan_amount"] = to_numeric_series(model_df["loan_amount"])
model_df["income"] = to_numeric_series(model_df["income"])
model_df["loan_to_value_ratio"] = to_numeric_series(model_df["loan_to_value_ratio"])
model_df["ltv_missing"] = model_df["loan_to_value_ratio"].isna().astype(int)
model_df["loan_to_value_ratio"] = model_df["loan_to_value_ratio"].fillna(model_df["loan_to_value_ratio"].median())
model_df["log_loan_amount"] = np.log1p(model_df["loan_amount"].clip(lower=0))
model_df["log_income"] = np.log1p(model_df["income"].clip(lower=0))

for col in ["loan_purpose", "loan_type", "applicant_sex", "occupancy_type", "lien_status", "applicant_age"]:
    model_df[col] = model_df[col].astype(str).str.strip()

model_df[["target_approved", "state_code", "dti_ordinal", "log_loan_amount", "log_income", "loan_to_value_ratio", "ltv_missing"]].head()

In [ ]:
state_covariate_columns = ["state_median_income", "lender_concentration_hhi"]

if not set(state_covariate_columns).issubset(model_df.columns):
    if STATE_COVARIATES_PATH.exists():
        state_cov = pd.read_csv(STATE_COVARIATES_PATH)
        state_cov["state_code"] = state_cov["state_code"].astype(str).str.upper().str.strip()
        model_df = model_df.merge(state_cov, on="state_code", how="left")
    else:
        raise FileNotFoundError("Add state_median_income and lender_concentration_hhi to the dataset or provide data/external/state_covariates.csv.")

for col in state_covariate_columns:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")
    model_df[col] = model_df[col].fillna(model_df.groupby("state_code")[col].transform("median"))
    model_df[col] = model_df[col].fillna(model_df[col].median())
    model_df[col + "_z"] = (model_df[col] - model_df[col].mean()) / model_df[col].std(ddof=0)

model_input_columns = [
    "target_approved", "state_code", "dti_ordinal", "log_loan_amount", "log_income",
    "loan_to_value_ratio", "ltv_missing", "dti_missing", "loan_purpose", "loan_type",
    "applicant_sex", "occupancy_type", "lien_status", "applicant_age",
    "state_median_income_z", "lender_concentration_hhi_z"
]
model_input = model_df[model_input_columns].dropna().copy()
print(f"Rows available for multilevel modeling: {len(model_input):,}")
model_input.head()

## 3. Export the model input and run the R/lme4 script

The R script is stored at `multilevel/run_multilevel_lme4.R`. It writes model outputs to `outputs/multilevel/`.

In [ ]:
model_input_path = OUT_DIR / "multilevel_model_input.csv"
model_input.to_csv(model_input_path, index=False)
print(f"Wrote {model_input_path}")

In [ ]:
r_script_path = PROJECT_ROOT / "multilevel" / "run_multilevel_lme4.R"
try:
    completed = subprocess.run(["Rscript", str(r_script_path)], cwd=PROJECT_ROOT, capture_output=True, text=True, check=False)
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError("Rscript finished with a nonzero exit code.")
except FileNotFoundError:
    print("Rscript was not found. Run multilevel/run_multilevel_lme4.R manually from the repository root.")
except RuntimeError as exc:
    print(exc)

## 4. Load generated model artifacts

These CSV files are the audit trail for the multilevel values reported in the paper.

In [ ]:
artifact_paths = {
    "table1": OUT_DIR / "table1_null_model.csv",
    "table2": OUT_DIR / "table2_full_random_intercept.csv",
    "slope": OUT_DIR / "random_slope_lrt_summary.csv",
    "eb": OUT_DIR / "figure9_eb_residuals.csv"
}

for name, path in artifact_paths.items():
    print(f"{name}: {'FOUND' if path.exists() else 'missing'} -> {path}")

if artifact_paths["table1"].exists():
    table1 = pd.read_csv(artifact_paths["table1"])
    display(table1)

if artifact_paths["table2"].exists():
    table2 = pd.read_csv(artifact_paths["table2"])
    display(table2.head(25))

if artifact_paths["slope"].exists():
    slope = pd.read_csv(artifact_paths["slope"])
    display(slope)

## 5. Compare model outputs to the report targets

This table makes it easy to check whether the fitted model output matches the values reported in the final paper.

In [ ]:
reported_targets = pd.DataFrame([
    {"artifact": "Table 1", "quantity": "Between-state variance", "reported": 0.312},
    {"artifact": "Table 1", "quantity": "ICC", "reported": 0.087},
    {"artifact": "Table 1", "quantity": "Log-likelihood", "reported": -191842},
    {"artifact": "Table 2", "quantity": "Residual between-state variance", "reported": 0.184},
    {"artifact": "Random slope", "quantity": "LRT chi-square", "reported": 18.4},
    {"artifact": "Random slope", "quantity": "DTI slope variance", "reported": 0.041},
    {"artifact": "Random slope", "quantity": "Cross-level interaction gamma", "reported": 0.089}
])
reported_targets.to_csv(OUT_DIR / "reported_multilevel_targets.csv", index=False)
reported_targets

## 6. Recreate Figure 9 from EB residuals

In [ ]:
if artifact_paths["eb"].exists():
    eb = pd.read_csv(artifact_paths["eb"]).sort_values("eb_residual")
    fig, ax = plt.subplots(figsize=(8, 11))
    colors = np.where(eb["eb_residual"] >= 0, "tab:blue", "tab:red")
    ax.barh(eb["state_code"], eb["eb_residual"], color=colors)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("EB residual: log-odds above/below predicted")
    ax.set_ylabel("State")
    ax.set_title("Empirical Bayes State Residuals")
    fig.tight_layout()
    fig_path = OUT_DIR / "fig09_eb_residuals_recreated.png"
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved {fig_path}")
else:
    print("EB residual file not found. Run the multilevel model script first.")